# GeoLab Tutorial: Raster Calculator — Multiply a DEM by a Constant Value

This notebook demonstrates how to apply a mathematical formula to every pixel in a raster file (e.g., a Digital Elevation Model) and export the result as a new GeoTIFF. It is the Python equivalent of QGIS's **Raster Calculator** tool.

**Output:** A new GeoTIFF file (`output_map.tif`) where each pixel value equals the original DEM elevation multiplied by a user-defined constant expression (here: `elevation × 0.5 × 100 × 9.8`).

> **Target audience:** GIS/QGIS users who are beginning to learn geospatial programming in Python.

## Requirements

Install the required packages before running this notebook:

```bash
pip install gdal numpy rasterio
```

Or using conda (recommended for GDAL on Windows):

```bash
conda install -c conda-forge gdal numpy rasterio
```

| Package | Purpose |
|---|---|
| `gdal` (osgeo) | Reading raster metadata and band data |
| `numpy` | Fast pixel-wise array arithmetic |
| `rasterio` | Writing the result back to a GeoTIFF file |

## Step 1: Import Libraries

In [ ]:
# GDAL (Geospatial Data Abstraction Library) is the backbone of geospatial I/O in Python.
# We use it here to open the input raster and extract its spatial metadata (CRS, geotransform).
from osgeo import gdal

# NumPy lets us treat the raster as a 2D array so we can apply math to all pixels at once,
# which is far faster than looping over every pixel individually.
import numpy as np

# Rasterio provides a clean, Pythonic API for writing raster files.
# We use it to save the calculated result as a new GeoTIFF.
import rasterio

# Suppress the GDAL 4.0 FutureWarning about exception handling defaults.
# Calling UseExceptions() is the recommended practice going forward.
gdal.UseExceptions()

## Step 2: Open the Input Raster and Read Its Metadata

We open the DEM file in **read-only** mode to prevent accidental modification of the source data. The GeoTransform encodes the raster's spatial location and pixel size — we need this to georeference the output file correctly.

In [ ]:
# --- USER CONFIGURATION ---
# Update this path to point to your own DEM (or any single-band raster).
INPUT_RASTER_PATH = "D:/download/Tutorial Data/Tutorial Data/Bompon/dem.tif"
# --------------------------

# Open the raster file using GDAL. gdal.GA_ReadOnly ensures we don't accidentally modify the source.
dem_raster = gdal.Open(INPUT_RASTER_PATH, gdal.GA_ReadOnly)

if dem_raster is None:
    raise FileNotFoundError(f"Could not open raster file: {INPUT_RASTER_PATH}")

# GetGeoTransform() returns a 6-element tuple describing the spatial reference of the raster:
# (x_top_left, pixel_width, rotation_x, y_top_left, rotation_y, pixel_height)
# We preserve this so the output file is placed correctly on the map.
geo_transform = dem_raster.GetGeoTransform()

print(f"Raster size : {dem_raster.RasterXSize} cols × {dem_raster.RasterYSize} rows")
print(f"Number of bands : {dem_raster.RasterCount}")
print(f"GeoTransform : {geo_transform}")

## Step 3: Read the Elevation Band into a NumPy Array

A DEM typically contains a single band where each pixel stores an elevation value. We extract **Band 1** and load it into memory as a 2D NumPy array so we can perform arithmetic on it.

In [ ]:
# GetRasterBand(1) accesses the first (and usually only) band of the DEM.
# Raster bands are 1-indexed in GDAL (unlike Python's 0-indexed arrays).
elevation_band = dem_raster.GetRasterBand(1)

# ReadAsArray() converts the band into a 2D NumPy array with shape (rows, cols).
# This is the key step that lets us do fast, vectorised math on pixel values.
elevation_array = elevation_band.ReadAsArray()

# Store the array dimensions — we'll need them to create the output array and for iteration.
n_rows, n_cols = elevation_array.shape

print(f"Array shape  : {elevation_array.shape}  (rows, cols)")
print(f"Data type    : {elevation_array.dtype}")
print(f"Min elevation: {elevation_array.min():.2f}")
print(f"Max elevation: {elevation_array.max():.2f}")

## Step 4: Apply the Raster Calculation

This is the core operation — equivalent to writing a formula in the QGIS Raster Calculator.

**Formula applied to each pixel:** `result = elevation × 0.5 × 100 × 9.8`

Two approaches are shown: a **vectorised** (recommended, fast) approach and a **loop-based** (educational, slow) approach. The vectorised version should always be preferred in practice.

In [ ]:
# --- Define the multiplier constant ---
# Adjust this expression to match your own formula.
# Example: 0.5 * 100 * 9.8 converts elevation to potential energy per unit mass (J/kg).
CONSTANT_MULTIPLIER = 0.5 * 100 * 9.8

# --- Vectorised approach (RECOMMENDED) ---
# NumPy applies the multiplication to every pixel simultaneously.
# This is orders of magnitude faster than a nested for-loop for large rasters.
result_array = elevation_array * CONSTANT_MULTIPLIER

print(f"Calculation complete.")
print(f"Result min : {result_array.min():.2f}")
print(f"Result max : {result_array.max():.2f}")

# --- Loop-based approach (EDUCATIONAL — shows the logic cell-by-cell) ---
# Uncomment the block below if you want to see the pixel-by-pixel equivalent.
# WARNING: This will be very slow on large rasters.
#
# result_array_loop = np.zeros(shape=(n_rows, n_cols))
# for i in range(n_rows):
#     for j in range(n_cols):
#         result_array_loop[i, j] = elevation_array[i, j] * CONSTANT_MULTIPLIER

## Step 5: Export the Result as a GeoTIFF

We write the calculated array back to disk as a georeferenced GeoTIFF, preserving the original CRS and spatial extent so it lines up correctly when loaded into QGIS or any other GIS software.

In [ ]:
def export_array_to_geotiff(
    array: np.ndarray,
    output_path: str,
    geo_transform: tuple,
    crs: str = "EPSG:4326"
) -> None:
    """
    Write a 2D NumPy array to a single-band GeoTIFF file.

    Parameters
    ----------
    array : np.ndarray
        2D array of pixel values to write (shape: rows × cols).
    output_path : str
        File path for the output GeoTIFF (e.g., 'output_map.tif').
    geo_transform : tuple
        GDAL GeoTransform tuple from the source raster, used to georeference the output.
    crs : str
        Coordinate Reference System in EPSG format (default: 'EPSG:4326' = WGS 84).

    Returns
    -------
    None
        The file is written to disk. A confirmation message is printed on success.
    """
    # Cast to float32 to keep file size reasonable while preserving decimal precision.
    # float64 doubles the file size with no benefit for most raster operations.
    array_float32 = np.float32(array)

    # rasterio.open() in write mode ('w') creates the output file.
    # We must specify the driver, dimensions, band count, data type, CRS, and transform.
    with rasterio.open(
        output_path,
        mode="w",
        driver="GTiff",          # Output format: GeoTIFF
        height=array_float32.shape[0],  # Number of rows
        width=array_float32.shape[1],   # Number of columns
        count=1,                 # Single band output
        dtype=np.float32,        # Pixel data type
        crs=crs,                 # Coordinate Reference System
        transform=rasterio.transform.from_gdal(*geo_transform),  # Spatial transform
    ) as output_file:
        # write() takes the array and the band index (1-indexed) to write to.
        output_file.write(array_float32, 1)

    print(f"Output saved to: {output_path}")


# --- USER CONFIGURATION ---
OUTPUT_RASTER_PATH = "output_map.tif"  # Change to your desired output path
OUTPUT_CRS = "EPSG:4326"               # Match the CRS of your input data
# --------------------------

export_array_to_geotiff(
    array=result_array,
    output_path=OUTPUT_RASTER_PATH,
    geo_transform=geo_transform,
    crs=OUTPUT_CRS
)

## Step 6: Verify the Output

Quick sanity check — re-open the saved file and confirm its dimensions and value range match expectations before loading it into QGIS.

In [ ]:
# Re-open the saved GeoTIFF to verify it was written correctly.
# This mimics what QGIS would see when you drag the file into the Layers panel.
with rasterio.open(OUTPUT_RASTER_PATH) as verification_file:
    saved_array = verification_file.read(1)  # Read band 1
    print("=== Output Raster Verification ===")
    print(f"File path  : {OUTPUT_RASTER_PATH}")
    print(f"CRS        : {verification_file.crs}")
    print(f"Shape      : {saved_array.shape}  (rows × cols)")
    print(f"Data type  : {saved_array.dtype}")
    print(f"Min value  : {saved_array.min():.4f}")
    print(f"Max value  : {saved_array.max():.4f}")
    print(f"Transform  : {verification_file.transform}")
    print("\n✅ Output file looks correct! Load it in QGIS to visualise the result.")